In [1]:
!pip install -q llama-index

In [2]:
import sys
import os
from pathlib import Path
import time
import logging
import numpy as np
import pickle
from llama_index.core import Document

# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))

from src.utils.index_classes import SparseBM25Vectorizer, SparseBM25Retriever, TreeNode
from src.utils.data_utils import select_random_documents
from src.utils.evaluation import (
    perform_analysis,
    perform_single_match_analysis,
    print_single_match_analysis,
    print_sample_results,
    calculate_prefix_success_at_k,
    perform_prefix_match_analysis,
    print_prefix_match_analysis
)

# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))

In [3]:
def load_tree_index(tree_index_path):
    """
    Load the tree index from the specified path.
    """
    print(f"Loading tree index from {tree_index_path}")
    with open(tree_index_path, 'rb') as f:
        index_data = pickle.load(f)
    print("Loaded tree index.")

    return index_data # (tree, vectorizer, documents)

def load_index(index_path):
    """
    Load the sparse index from the specified path
    """
    print(f"Loading index from {index_path}")
    with np.load(index_path / "embeddings.npz", 'rb') as data:
        embeddings = {key: data[key] for key in data.files}
    with open(index_path / "vectorizer.pkl", 'rb') as f:
        vectorizer = pickle.load(f)
    with open(index_path / "documents.pkl", 'rb') as f:
        documents = pickle.load(f)
    
    print("Loaded index.")
        
    return embeddings, vectorizer, documents

def level_based_search(query, tree, vectorizer, top_k=10):
    """
    Search through a hierarchical tree structure based on query similarity.
    
    Args:
        query (str): The search query text
        tree (TreeNode): The root node of the tree
        vectorizer: An object with a transform method to convert text to vectors
        top_k (int): Number of results to return
    
    Returns:
        list: List of paths, where each path is a list of (node, score, is_selected) tuples
    """
    # Transform the query to vector form for similarity comparison
    query_embedding = vectorizer.transform([query])
    
    def compute_similarity(node):
        """Compute similarity between query and node"""
        if not hasattr(node, 'embedding') or node.embedding is None:
            return 0.0
        
        # Compute dot product (BM25 similarity)
        similarity = query_embedding @ node.embedding.T
        similarity = float(similarity.toarray()[0][0]) if hasattr(similarity, 'toarray') else float(similarity)
        
        return similarity
    
    def count_leaf_nodes(node):
        """Count the number of leaf nodes in a subtree."""
        if not node.children:
            return 1
        
        count = 0
        for child in node.children.values():
            count += count_leaf_nodes(child)
        
        return count
    
    def get_max_tree_depth(node, current_depth=0):
        """Get the maximum depth of the tree starting from the given node."""
        if not node.children:
            return current_depth
        
        max_depth = current_depth
        for child in node.children.values():
            child_depth = get_max_tree_depth(child, current_depth + 1)
            max_depth = max(max_depth, child_depth)
        
        return max_depth
    
    # Get the maximum depth of the tree
    max_depth = get_max_tree_depth(tree)
    
    # Start with the root as our initial path
    initial_path = [(tree, compute_similarity(tree), True)]
    active_paths = [initial_path]
    
    # For each level in the tree
    for level in range(max_depth):
        next_level_paths = []
        
        # For each active path from the previous level
        for path in active_paths:
            # Get the last node in the path, which should be at the current level
            last_node, _, is_selected = path[-1]
            
            # Only proceed if this node was selected
            if not is_selected:
                next_level_paths.append(path)
                continue
            
            # If this is a leaf node, add it to the next level and continue
            if not last_node.children:
                next_level_paths.append(path)
                continue
            
            # Get all children at this level with their similarity scores
            children = list(last_node.children.values())
            child_scores = [(child, compute_similarity(child)) for child in children]
            
            # Sort children by similarity score in descending order
            child_scores.sort(key=lambda x: x[1], reverse=True)
            
            # Group children by similarity to handle ties (rounded to 6 decimal places)
            from collections import defaultdict
            score_groups = defaultdict(list)
            for child, score in child_scores:
                rounded_score = round(score, 6)
                score_groups[rounded_score].append((child, score))
            
            # Get scores in descending order
            scores = sorted(score_groups.keys(), reverse=True)
            
            # Start with the highest scoring group
            selected_children = []
            remaining_children = []
            
            # First, try to select all children from the highest scoring group
            highest_score = scores[0]
            highest_group = score_groups[highest_score]
            
            # Add all children from the highest group as selected
            for child, score in highest_group:
                selected_children.append((child, score, True))
            
            # Count total leaves under selected children
            total_leaves = sum(count_leaf_nodes(child) for child, _, _ in selected_children)
            
            # If we need more leaves, add more children in descending score order
            score_index = 1
            while total_leaves < top_k and score_index < len(scores):
                current_score = scores[score_index]
                current_group = score_groups[current_score]
                
                for child, score in current_group:
                    selected_children.append((child, score, True))
                    total_leaves += count_leaf_nodes(child)
                    
                    # If we have enough leaves, break
                    if total_leaves >= top_k:
                        break
                
                score_index += 1
            
            # Add any remaining children as non-selected
            for score in scores[score_index:]:
                for child, child_score in score_groups[score]:
                    remaining_children.append((child, child_score, False))
            
            # Create new paths for each selected child
            for child, score, is_selected in selected_children:
                new_path = path.copy()
                new_path.append((child, score, is_selected))
                next_level_paths.append(new_path)
            
            # Create paths for non-selected children as well
            for child, score, is_selected in remaining_children:
                new_path = path.copy()
                new_path.append((child, score, is_selected))
                next_level_paths.append(new_path)
        
        # Update active paths for the next iteration
        active_paths = next_level_paths
    
    # Filter out paths that don't end in leaf nodes
    leaf_paths = [path for path in active_paths if not path[-1][0].children]
    
    # If we don't have enough leaf paths, take what we have
    if len(leaf_paths) < top_k:
        results = leaf_paths
    else:
        # Create a key function for sorting paths
        def path_sort_key(path):
            # Extract (level, score) pairs for each level's selected node
            level_scores = []
            for i, (node, score, is_selected) in enumerate(path):
                if is_selected:
                    level_scores.append((i, score))
            
            # Return a tuple of scores ordered by level for lexicographical sorting
            # Negate scores so higher scores come first
            return tuple(-score for _, score in sorted(level_scores, key=lambda x: x[0]))
        
        # Sort paths by the scores at each level
        leaf_paths.sort(key=path_sort_key)
        
        # Take the top_k paths
        results = leaf_paths[:top_k]
    
    return results

def best_path_search(query, tree, vectorizer, top_k=5):
    """
    Search for the best paths through the tree based on similarity at each level.
    """
    if not query:
        print("Empty query.")
        return None
    
    #print(f"Searching for: '{query}' (top {top_k} results)")
    
    # Find best paths through the tree
    best_paths = level_based_search(query, tree, vectorizer, top_k)
    
    return best_paths

def extract_leaf_scores_and_text(best_paths, documents):
    leaf_data = []
    
    # Create a fast lookup table for documents by item_key
    document_lookup = {doc.metadata.get('item_key'): doc for doc in documents if hasattr(doc, 'metadata')}
    
    for path in best_paths:

        # Get all scores from the path in top-down order
        scores = [score for _, score, _ in path]
        
        # Get the last node from the path (leaf node)
        leaf_node = path[-1][0]
        
        # Get item_key from leaf_node
        item_key = getattr(leaf_node, 'item_key', None)
        if item_key and item_key in document_lookup:
            doc = document_lookup[item_key]
            text = doc.text if hasattr(doc, 'text') else "No text available"
            
            leaf_data.append({
                'item_key': item_key,
                'text': text,
                'score': scores                
            })
    
    return leaf_data
    

In [4]:
def main(file, batch_size=500, top_k=10):

    start_total =  time.time()
    # Load the tree 'texto' index
    index_path_texto = Path('/work/data/indexes/' + file + '_Tree_bm25sparseindex')
    tree_index_filepath = index_path_texto / "tree_index.pkl"
    tree, tree_vectorizer, tree_documents = load_tree_index(tree_index_filepath)
    
    # Load 'resumen' documents and embeddings from BM25, for queries
    bm25_path_resumen = Path('/work/data/indexes/' + file + '_resumen_BM25sparseindex')
    bm25_embeddings, bm25_vectorizer, bm25_documents = load_index(bm25_path_resumen)
    
    # Select a random subset of queries for evaluation.
    sample_size = min(len(bm25_documents), 16590)
    selected_docs = select_random_documents(bm25_documents, sample_size) # [('item_key', 'resumen'), ...]
    total_queries = len(selected_docs)
    all_results = []
    total_batches = (total_queries + batch_size - 1) // batch_size
    
    # Starts processing in batches
    start_batches = time.time()
    
    for batch_num, start_idx in enumerate(range(0, total_queries, batch_size), 1):
        end_idx = min(start_idx + batch_size, total_queries)
        batch = selected_docs[start_idx:end_idx]
        for doc_key, doc_text in batch:
            query = doc_text
            # Search the tree index
            results = best_path_search(query, tree, tree_vectorizer, top_k) # [[(TreeNode, score, True), (...)], ...]
    
            # Get target score from matching query with texto documents by item_key
            query_embedding = tree_vectorizer.transform([query])
            query_item_key = doc_key
            document_dict = {doc.metadata.get('item_key'): doc for doc in tree_documents} # Create a dict for fast lookup
            target_text = document_dict.get(query_item_key).text_resource.text if query_item_key in document_dict else None
            target_embedding = tree_vectorizer.transform([target_text])
            target_score = query_embedding @ target_embedding.T
            target_score = float(target_score.toarray()[0][0]) if hasattr(target_score, 'toarray') else float(target_score)
    
            # Append results for this resumen document
            all_results.append({
                'resumen_item_key': query_item_key,
                'resumen_text': query,
                'target_text': target_text,
                'target_score': target_score,
                'texto_results': extract_leaf_scores_and_text(results, tree_documents)
            })
    
            # Logging progress for each batch
            elapsed_batches = time.time() - start_batches
            batches_remaining = total_batches - batch_num
            avg_batch_time = elapsed_batches / batch_num
            estimated_remaining_sec = batches_remaining * avg_batch_time
            est_min, est_sec = divmod(estimated_remaining_sec, 60)
            logging.info(
                f"Batch {batch_num}/{total_batches} processed. "
                f"Estimated time to completion: {int(est_min)}m {int(est_sec)}s"
            )
    
    # Perform overall analysis
    analysis_results = perform_analysis(all_results)
    print("\nAnalysis Results:")
    for metric, value in analysis_results.items():
        print(f"{metric}: {value:.4f}")
        
    # Perform single-match analysis and print results
    metrics = perform_single_match_analysis(all_results, top_k=top_k)
    print_single_match_analysis(metrics, top_k=top_k)

    # Perform prefix-matching analysis
    prefix_length = 6  # Match first 6 characters of item_key
    prefix_match_metrics = perform_prefix_match_analysis(
        all_results, 
        top_k=10, 
        prefix_length=prefix_length
    )
    print_prefix_match_analysis(
        prefix_match_metrics, 
        top_k=10, 
        prefix_length=prefix_length
    )

    # Print some sample retrieval results
    print_sample_results(all_results)

    total_elapsed = time.time() - start_total
    tot_min, tot_sec = divmod(total_elapsed, 60)
    print(f"\nTotal execution time: {int(tot_min)}m {int(tot_sec)}s")

In [5]:
main('OEB')

Loading tree index from /work/data/indexes/OEB_Tree_bm25sparseindex/tree_index.pkl
Loaded tree index.
Loading index from /work/data/indexes/OEB_resumen_BM25sparseindex
Loaded index.

Analysis Results:
Recall: 0.0410
Precision: 0.0041
F1 Score: 0.0075
Mean Reciprocal Rank (MRR): 0.0265
Mean Average Precision (MAP): 0.0265

Single-Match Analysis Results:
Success@10             : 4.10%
MRR: 0.0265
Mean Position: 2.82
Median Position: 2.0
Perfect Matches: 2.01%

Position Distribution:
------------------------------
Position_1: 2.01%
Position_2: 0.77%
Position_3: 0.15%
Position_4: 0.33%
Position_5: 0.19%
Position_6: 0.06%
Position_7: 0.23%
Position_8: 0.07%
Position_9: 0.17%
Position_10: 0.12%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 4.40%
Prefix_MRR: 0.0380
Prefix_Mean_Position: 1.78
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 3.59%

Prefix Position Distribution:
----------------------------------------
Position 1    : 3.59%
Position 2